**Попытка попробовать другую модель # 1**

Здесь я решила попробовать еще один вид матричных разложений, а конкретнее SVD (singular value decomposition), с которым мы работали на одном из уроков по рекомендательным системам. Для обучения использовала уже обработанные при обучении ALS данные (см.самый первый ноутбук EDA + ALS, где показано, на основе чего я их получила) и ряд написанных ранее показателей. Модель предсказывает чуть хуже обычного ALS,объединение двух моделей также не сильно повлияло на общий результат. Поэтому я решила попробовать объединить модели матричного разложения с SVD и ALS. Результаты этой работы можно увидеть в других ноутбуках, приложенных к сданным материалам.

In [1]:
import os
import csv
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from textwrap import wrap
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD, SlopeOne, BaselineOnly
from sklearn.preprocessing import MinMaxScaler
from surprise.model_selection import cross_validate, PredefinedKFold, GridSearchCV
from itertools import product
import tensorflow as tf

2025-02-25 16:11:05.581069: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-25 16:11:05.581104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-25 16:11:05.581620: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-25 16:11:05.584907: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-25 16:11:06.018874: W tensorflow/compiler/tf2

In [63]:
#Напишем базовую комплектацию показателей для проверки точности вычисления рейтинга моделью.
def Avg_Precision_at_n(fact, predicted, n=10):
    """
    Вычисление средней точности по n-позициям для двух списков значений.
    
    Вход
    ----------
    fact : list
             Фактический список элементов, который нужно предсказать.
    predicted : list
             Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
             Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Средняя точность на n-позициях
    """
    if not fact:
        return 0.0

    if len(predicted)>n:
        predicted = predicted[:n]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # Первое условие проверяет наличие предсказания в списке фактических элементов
        # второе условие - проверка на отсутствие (или наличие) повторов в предсказании
        if p in fact and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    return score / min(len(fact), n)

def MAP_at_n(fact, predicted, n=10):
    """
    Вычисление mean average precision at n для двух списков элементов.
   
    Вход
    ----------
    fact : list
            Фактический список элементов, который нужно предсказать.
    predicted : list
            Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
            Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Mean average precision at n двух списков
    """
    return np.mean([Avg_Precision_at_n(f,p,n) for f,p in zip(fact, predicted)])

In [3]:
#Для подачи в модели:
files_dir = './data/recsys/'

TRAIN_CSV_PATH = os.path.join(files_dir, 'train_df.csv')
TEST_CSV_PATH = os.path.join(files_dir, 'test_df.csv')

reader = Reader(line_format="user item rating", sep = '\t',rating_scale=(1, 10)) # Зададим разброс оценок

folds_files = [(TRAIN_CSV_PATH,TEST_CSV_PATH)] #список путей к файлам для подачи в объект библиотеки surprise lib
data = Dataset.load_from_folds(folds_files, reader=reader) #создадим data-объект

pkf = PredefinedKFold() #создадим объект, позволяющий подать в модель собственный набор train и test данных
trainset, testset = next(pkf.split(data)) #определим train и test сеты

In [4]:
#Для поиска лучших опций загрузим из файла coef_df:
coef_df = pd.read_csv(TRAIN_CSV_PATH, sep='\t')
#Для передачи в модель:
data_coef = Dataset.load_from_df(coef_df, reader=reader) #создадим data-объект

In [5]:
# param_grid = {
#     "n_factors":[1, 5, 10, 15, 20, 30, 40, 50, 100],
#     "n_epochs": [5, 10, 15, 20, 30, 40, 50, 100],
#     "lr_all": [0.001, 0.002, 0.005],
#     "reg_all": [0.02, 0.08, 0.4, 0.6]
# }


# gs = GridSearchCV(SVD, param_grid, measures=["rmse"], refit=True, cv=3)

# gs.fit(data_coef)

# training_parameters = gs.best_params["rmse"]

# print("BEST RMSE: \t", gs.best_score["rmse"])
# print("BEST params: \t", gs.best_params["rmse"])

In [6]:
#Предыдущий вариант оказался очень долгим, поэтому чуть поменяем подход:
# n_factors = [1, 5, 10, 15, 20, 30, 40, 50, 100]
# lr_all = [0.001, 0.002, 0.005]
# reg_all = [0.02, 0.08, 0.1, 0.2, 0.4]
# best_score = 2
# for opts in list(product(n_factors,lr_all,reg_all)):
#     print('Checking model with options: ', opts) #для понимания этапа процесса
#     algo = SVD(n_factors=opts[0],lr_all = opts[1],reg_all = opts[2],
#                random_state=999, verbose=False)
#     cv=cross_validate(algo, data_coef, measures=['RMSE'], cv=3)
#     curr_score = np.mean(cv['test_rmse'])
#     if curr_score < best_score:
#         best_score = curr_score
#         best_opts = str(opts)
# print('Best mean RMSE =', best_score, ' for SVD with options: ' ,str(best_opts))

*Best mean RMSE = 1.5928263381293533  for SVD with options:  (1, 0.002, 0.08)*
Попробуем посмотреть еще и на параметр n_epochs:

In [7]:
# Попробуем посмотреть еще и на параметр n_epochs:
# n_epochs = [10, 20, 30, 40, 50, 80, 100, 120, 150, 180, 200]
# best_score = 2
# for eps in n_epochs:
#     algo = SVD(n_epochs = eps, n_factors=1,lr_all = 0.002,reg_all = 0.08, random_state=999, verbose=False)
#     cv=cross_validate(algo, data_coef, measures=['RMSE'], cv=3)
#     curr_score = np.mean(cv['test_rmse'])
#     if curr_score < best_score:
#         best_score = curr_score
#         best_eps = eps
# print('Best mean RMSE =', best_score, ' for SVD with epochs: ', best_eps)

*Best mean RMSE = 1.5928263381293533  for SVD with epochs:  20*

In [8]:
#Попробуем обучить SVD модель с лучшим результатом:
algo = SVD(n_epochs = 20, n_factors=1, lr_all = 0.002,reg_all = 0.08, random_state=999, verbose=False)

In [9]:
%%time
#Сделаем предсказание:
predictions_SVD = algo.fit(trainset).test(testset)

CPU times: user 8.66 s, sys: 44.4 ms, total: 8.71 s
Wall time: 8.71 s


In [10]:
#Соберем в сет:
appended_data_SVD = []
for i in predictions_SVD:
    appended_data_SVD.append(i)
pred_svd = pd.DataFrame(appended_data_SVD, columns = ['user_id','product_id','real_rating','predicted_svd','details'])
pred_svd.drop(columns = ["details"], inplace = True)

In [11]:
pred_svd["user_id"] = pred_svd.user_id.astype(np.int32)
#Теперь нам нужно правильно отсортировать предсказание по столбцу с предполагаемым рейтингом, обрезав его до 10 значений:
svd_grouped = pred_svd.merge(pred_svd
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').predicted_svd.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
#Проверка
svd_grouped.head(15)

,user_id,product_id,real_rating,predicted_svd
0,1,38928,7.0,4.090568
1,1,46149,9.0,3.955564
2,1,196,10.0,3.952338
3,1,39657,8.0,3.334004
4,1,35951,4.0,3.318374
5,1,12427,2.0,3.134304
6,1,25133,6.0,2.938869
7,1,10258,5.0,2.932837
8,1,13032,3.0,2.493631
9,2,24852,10.0,4.524517


In [12]:
#Соберем все id продуктов в один столбец - pred_order:
svd_result = svd_grouped.groupby('user_id')['product_id'].unique().reset_index()
svd_result.columns=['user_id','pred_order']
#Check
svd_result.head(15)

,user_id,pred_order
0,1,"[38928, 46149, 196, 39657, 35951, 12427, 25133..."
1,2,"[24852, 47209, 21709, 18523, 33754, 1559, 7781..."
2,3,"[47766, 21903, 39190, 17668, 43961]"
3,7,"[40852, 21137, 37602, 47272, 31683, 30391, 276..."
4,13,"[4210, 27086, 27435, 43086, 41926, 42248, 1419..."
5,14,"[29509, 15172, 15869, 38845, 23803, 37266, 111..."
6,15,"[196, 48142]"
7,17,"[14146, 7350, 18534, 16797, 48933, 9006, 18567..."
8,21,"[49235, 18523, 32645, 28465, 28204, 17982, 247..."
9,22,"[35221, 24964, 7948, 24506]"


In [13]:
#Таким же образом соберем факт:
fact_df = pred_svd.merge(pred_svd
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').real_rating.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
#Проверка
fact_df.head(15)

,user_id,product_id,real_rating,predicted_svd
0,1,196,10.0,3.952338
1,1,46149,9.0,3.955564
2,1,39657,8.0,3.334004
3,1,38928,7.0,4.090568
4,1,25133,6.0,2.938869
5,1,10258,5.0,2.932837
6,1,35951,4.0,3.318374
7,1,13032,3.0,2.493631
8,1,12427,2.0,3.134304
9,2,24852,10.0,4.524517


In [14]:
fact_df = fact_df.groupby('user_id')['product_id'].unique().reset_index()
fact_df.columns=['user_id','fact_order']
#Check
fact_df.head(15)

,user_id,fact_order
0,1,"[196, 46149, 39657, 38928, 25133, 10258, 35951..."
1,2,"[24852, 16589, 1559, 19156, 18523, 22825, 2741..."
2,3,"[39190, 47766, 21903, 43961, 17668]"
3,7,"[47272, 29993, 31683, 27690, 9598, 13198, 3039..."
4,13,"[27435, 27086, 4210, 43086, 34382, 41926, 1419..."
5,14,"[39399, 29509, 23803, 15869, 8744, 37266, 1113..."
6,15,"[196, 48142]"
7,17,"[7350, 18534, 9006, 26767, 14146, 16797, 18567..."
8,21,"[25740, 6576, 17982, 25513, 8214, 18523, 24799..."
9,22,"[35221, 24964, 7948, 24506]"


In [15]:
result_svd = fact_df.merge(svd_result, how = 'left')
result_svd.head()

,user_id,fact_order,pred_order
0,1,"[196, 46149, 39657, 38928, 25133, 10258, 35951...","[38928, 46149, 196, 39657, 35951, 12427, 25133..."
1,2,"[24852, 16589, 1559, 19156, 18523, 22825, 2741...","[24852, 47209, 21709, 18523, 33754, 1559, 7781..."
2,3,"[39190, 47766, 21903, 43961, 17668]","[47766, 21903, 39190, 17668, 43961]"
3,7,"[47272, 29993, 31683, 27690, 9598, 13198, 3039...","[40852, 21137, 37602, 47272, 31683, 30391, 276..."
4,13,"[27435, 27086, 4210, 43086, 34382, 41926, 1419...","[4210, 27086, 27435, 43086, 41926, 42248, 1419..."


In [16]:
#Применим функцию MAP_at_n построчно:
result_svd['MAP'] = result_svd.apply(lambda x: MAP_at_n(x.fact_order, x.pred_order), axis=1)
result_svd.head()

,user_id,fact_order,pred_order,MAP
0,1,"[196, 46149, 39657, 38928, 25133, 10258, 35951...","[38928, 46149, 196, 39657, 35951, 12427, 25133...",0.409753
1,2,"[24852, 16589, 1559, 19156, 18523, 22825, 2741...","[24852, 47209, 21709, 18523, 33754, 1559, 7781...",0.276500
2,3,"[39190, 47766, 21903, 43961, 17668]","[47766, 21903, 39190, 17668, 43961]",0.244667
3,7,"[47272, 29993, 31683, 27690, 9598, 13198, 3039...","[40852, 21137, 37602, 47272, 31683, 30391, 276...",0.233917
4,13,"[27435, 27086, 4210, 43086, 34382, 41926, 1419...","[4210, 27086, 27435, 43086, 41926, 42248, 1419...",0.568333


In [17]:
sum(result_svd.MAP)/len(result_svd.user_id.unique())

0.35570340785603816

Неплохая точность, чуть менее ALS-варианта с наилучшими параметрами. Попробуем объединить варианты и посмотреть, получится ли улучшить качество предсказания.

In [18]:
#Обучим ALS-модель с лучшими опциями:
bsl_options =  {'method': 'als', 'n_epochs': 20, 'reg_u': 18, 'reg_i': 6}
algo = BaselineOnly(bsl_options = bsl_options)
#Сделаем предсказание:
predictions_als = algo.fit(trainset).test(testset)

Estimating biases using als...


In [19]:
#Соберем в сет:
appended_data_als = []
for i in predictions_als:
    appended_data_als.append(i)
pred_als = pd.DataFrame(appended_data_als, columns = ['user_id','product_id','real_rating','predicted_als','details'])
pred_als.drop(columns = ["details"], inplace = True)

In [20]:
#Объединим предсказания:
pred_als["user_id"] = pred_als.user_id.astype(np.int32) #obj to int  для сопоставимости
pred_combo = pred_als.merge(pred_svd, how = 'left')
pred_combo.head(15)

,user_id,product_id,real_rating,predicted_als,predicted_svd
0,1,196,10.0,4.170846,3.952338
1,1,46149,9.0,4.080973,3.955564
2,1,39657,8.0,3.351688,3.334004
3,1,38928,7.0,4.115123,4.090568
4,1,25133,6.0,2.929239,2.938869
5,1,10258,5.0,2.933067,2.932837
6,1,35951,4.0,3.407510,3.318374
7,1,13032,3.0,2.488235,2.493631
8,1,12427,2.0,3.196634,3.134304
9,2,24852,10.0,4.943528,4.524517


In [21]:
#Простая комбинация предсказаний в пропорции 50 на 50:
pred_combo['predicted'] = (pred_combo.predicted_als + pred_combo.predicted_svd)/2

In [22]:
#Отсортируем предсказание по столбцу с предполагаемым рейтингом, обрезав его до 10 значений:
combo_grouped = pred_combo.merge(pred_combo
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').predicted.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся

#Соберем все id продуктов в один столбец - pred_order:
combo_result = combo_grouped.groupby('user_id')['product_id'].unique().reset_index()
combo_result.columns=['user_id','pred_order']
#Check
combo_result.head(15)

,user_id,pred_order
0,1,"[38928, 196, 46149, 35951, 39657, 12427, 25133..."
1,2,"[24852, 47209, 21709, 18523, 33754, 1559, 7781..."
2,3,"[47766, 21903, 39190, 17668, 43961]"
3,7,"[40852, 21137, 37602, 47272, 31683, 30391, 276..."
4,13,"[4210, 27086, 27435, 43086, 41926, 42248, 1419..."
5,14,"[29509, 15172, 15869, 38845, 23803, 37266, 874..."
6,15,"[196, 48142]"
7,17,"[14146, 7350, 18534, 16797, 48933, 9006, 26767..."
8,21,"[49235, 18523, 32645, 28465, 28204, 17982, 247..."
9,22,"[35221, 24964, 7948, 24506]"


In [23]:
#Добавим столбец fact_order:
result = fact_df.merge(combo_result, how = 'left')
#Посчитаем MAP
result['MAP'] = result.apply(lambda x: MAP_at_n(x.fact_order, x.pred_order), axis=1)
result.head()

,user_id,fact_order,pred_order,MAP
0,1,"[196, 46149, 39657, 38928, 25133, 10258, 35951...","[38928, 196, 46149, 35951, 39657, 12427, 25133...",0.354198
1,2,"[24852, 16589, 1559, 19156, 18523, 22825, 2741...","[24852, 47209, 21709, 18523, 33754, 1559, 7781...",0.276500
2,3,"[39190, 47766, 21903, 43961, 17668]","[47766, 21903, 39190, 17668, 43961]",0.244667
3,7,"[47272, 29993, 31683, 27690, 9598, 13198, 3039...","[40852, 21137, 37602, 47272, 31683, 30391, 276...",0.233917
4,13,"[27435, 27086, 4210, 43086, 34382, 41926, 1419...","[4210, 27086, 27435, 43086, 41926, 42248, 1419...",0.568333


In [24]:
sum(result.MAP)/len(result.user_id.unique())

0.3558402134733201

В результате мы чуть увеличили точность предсказния по сравнению с чистой версией SVD, но ALS с задачей справился всё-таки получше.